# Phần 1: Tiền xử lý dữ liệu (Data Preprocessing)

## 1.1. Tải dữ liệu

In [ ]:
# Import thư viện
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Thiết lập đường dẫn thư mục chứa dữ liệu đầu vào
DATA_DIR = 'dataset/'

> Sales : dữ liệu doanh thu (huấn luyện)

In [ ]:
df_sales = pd.read_csv(DATA_DIR + 'sales.csv', parse_dates=['Date'])
df_sales.info()

> Products : Danh mục sản phẩm

In [ ]:
df_products = pd.read_csv(DATA_DIR + 'products.csv')
df_products.info()

> Customers : Thông tin khách hàng

In [ ]:
df_customers = pd.read_csv(DATA_DIR + 'customers.csv', parse_dates=['signup_date'])
df_customers.info()

> Promotions : Các chiến dịch khuyến mãi

In [ ]:
df_promotions = pd.read_csv(DATA_DIR + 'promotions.csv', parse_dates=['start_date','end_date'])
df_promotions.info()

> Geography : Danh sách mã bưu chính các vùng

In [ ]:
df_geography = pd.read_csv(DATA_DIR + 'geography.csv')
df_geography.info()

> Orders : thông tin đơn hàng

In [ ]:
df_orders = pd.read_csv(DATA_DIR + 'orders.csv', parse_dates=['order_date'])
df_orders.info()

> Order Items : Chi tiết từng dòng sản phẩm trong đơn

In [ ]:
df_order_items = pd.read_csv(DATA_DIR + 'order_items.csv', low_memory=False)
df_order_items.info()

> Payments : Thông tin thanh toán (tương ứng 1:1 với đơn hàng)

In [ ]:
df_payments = pd.read_csv(DATA_DIR + 'payments.csv')
df_payments.info()

> Shipments : Thông tin vận chuyển

In [ ]:
df_shipments = pd.read_csv(DATA_DIR + 'shipments.csv', parse_dates=['ship_date','delivery_date'])
df_shipments.info()

> Returns : Các sản phẩm trả lại

In [ ]:
df_returns = pd.read_csv(DATA_DIR + 'returns.csv', parse_dates=['return_date'])
df_returns.info()

> Reviews : Đánh giá sản phẩm sau giao hàng

In [ ]:
df_reviews = pd.read_csv(DATA_DIR + 'reviews.csv', parse_dates=['review_date'])
df_reviews.info()

> Inventory : Ảnh chụp tồn kho cuối tháng

In [ ]:
df_inventory = pd.read_csv(DATA_DIR + 'inventory.csv', parse_dates=['snapshot_date'])
df_inventory.info()

> Web Traffic : Lưu lượng truy cập website mỗi ngày

In [ ]:
df_web_traffic = pd.read_csv(DATA_DIR + 'web_traffic.csv', parse_dates=['date'])
df_web_traffic.info()

## 1.2. Làm sạch và kiểm định chất lượng dữ liệu

In [ ]:
# Gom tất cả các bảng vào một dictionary để dễ dàng quét lỗi
all_dataframes = {
    'sales': df_sales, 'products': df_products, 'customers': df_customers, 'promotions': df_promotions, 'geography': df_geography, 'orders': df_orders, 'order_items': df_order_items, 'payments': df_payments, 'shipments': df_shipments, 'returns': df_returns, 'reviews': df_reviews, 'inventory': df_inventory, 'web_traffic': df_web_traffic
}

**Tính đầy đủ (Completeness)**

In [ ]:
# Quét qua từng bảng để tìm lỗi (nếu có)
for name, df in all_dataframes.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    total   = len(df)
    if not missing.empty:
        print(f"\n[!] Bảng '{name}' ({total:,} dòng) — CÓ dữ liệu bị thiếu:")
        for col, n in missing.items():
            pct = n / total * 100
            bar = '█' * int(pct / 5)
            print(f"      {col:<30} {n:>8,}  ({pct:5.1f}%)  {bar}")
    else:
        print(f"[✓] Bảng '{name}' ({total:,} rows)")

*Nhận xét:*
- Cột `applicable_category` của bảng 'promotions' cho phép `null` nếu áp dụng tất cả danh mục $\rightarrow$ Không cần phải xử lý.
- Cột `promo_id` và `promo_id_2` của bảng 'order_items' cho phép `null` $\rightarrow$ không cần phải xử lý.

**Tính chính xác (Accuracy)**

In [ ]:
# Kiểm tra giá trị âm (nếu có)
for name in ['sales', 'products', 'order_items', 'payments', 'shipments', 'returns', 'inventory', 'web_traffic']:
    print(f"\n>> Thống kê bảng '{name}':")
    display(all_dataframes[name].describe())

*Nhận xét:* Các dữ liệu đều không có sai số logic về tiền tệ hay số lượng.

**Tính hợp lệ (Validity)**

In [ ]:
# Dữ liệu dạng nhãn
categorical_map = {
    'customers': ['gender', 'age_group'],
    'products': ['category', 'segment'],
    'orders': ['order_status', 'payment_method', 'device_type']
}
for name, cols in categorical_map.items():
    for col in cols:
        print(f"Bảng {name} - Cột {col}: {all_dataframes[name][col].unique()}")

In [ ]:
# Ràng buộc trong 'products': cogs < price
invalid_products = df_products[df_products['cogs'] >= df_products['price']]
print("Số sản phẩm không hợp lệ:", len(invalid_products))

*Nhận xét:* Các giá trị trong các cột phân loại cho thấy dữ liệu đã được chuẩn hóa cực kỳ tốt, ví dụ như `gender` và `age_group`: các nhóm được chia rõ ràng, không có giá trị rác.

**Tính nhất quán (Consistency)**

In [ ]:
# Kiểm tra lỗi phân tích ngày (NaT)
for name, df in all_dataframes.items():
    date_cols = df.select_dtypes(include=['datetime64']).columns
    for col in date_cols:
        nat_count = df[col].isna().sum()
        if nat_count > 0: print(f"[!] {name}.{col} có {nat_count} dòng lỗi định dạng ngày!")
print('[✓] Không có lỗi định dạng ngày.')

In [ ]:
# Kiểm tra tính toàn vẹn của Khóa ngoại
fk_relationships = [
    (df_customers, 'zip', df_geography, 'zip'),
    (df_orders, 'customer_id', df_customers, 'customer_id'),
    (df_orders, 'zip', df_geography, 'zip'),
    (df_order_items, 'order_id', df_orders, 'order_id'),
    (df_order_items, 'product_id', df_products, 'product_id'),
    (df_order_items, 'promo_id', df_promotions, 'promo_id'),
    (df_order_items, 'promo_id_2', df_promotions, 'promo_id'),
    (df_payments, 'order_id', df_orders, 'order_id'),
    (df_shipments, 'order_id', df_orders, 'order_id'),
    (df_returns, 'order_id', df_orders, 'order_id'),
    (df_returns, 'product_id', df_products, 'product_id'),
    (df_reviews, 'order_id', df_orders, 'order_id'),
    (df_reviews, 'product_id', df_products, 'product_id'),
    (df_reviews, 'customer_id', df_customers, 'customer_id'),
    (df_inventory, 'product_id', df_products, 'product_id')
]
for df1, col1, df2, col2 in fk_relationships:
    orphans = df1[~df1[col1].isin(df2[col2])][col1].nunique()
    print(f"Liên kết: {col1} -> {col2} | Số bản ghi lỗi: {orphans}")

*Nhận xét:* Dữ liệu không bị trùng lặp, đảm bảo các tính toán không bị thổi phồng.

**Tính duy nhất (Uniqueness)**

In [ ]:
for name, df in all_dataframes.items():
    dupes = df.duplicated().sum()
    print(f"Bảng '{name}': {dupes} dòng trùng lặp.")

*Nhận xét:* Các bộ dữ liệu ở mỗi cơ sở dữ liệu là duy nhất.

## 1.3. Hợp nhất dữ liệu và kỹ thuật tạo đặc trưng

In [ ]:
# Hợp nhất 
master = df_order_items.merge(df_products, on='product_id', how='left')
master = master.merge(df_orders, on='order_id', how='left')
master = master.merge(df_customers.drop(columns=['zip', 'city']), on='customer_id', how='left')
master = master.merge(df_geography, on='zip', how='left')

# Tính toán các cột chỉ số quan trọng (Derived Features)
# 1. Doanh thu thực tế (sau khi trừ chiết khấu)
master['total_revenue'] = (master['unit_price'] * master['quantity']) - master['discount_amount']

# 2. Giá vốn hàng bán (COGS)
master['total_cogs'] = master['cogs'] * master['quantity']

# 3. Lợi nhuận gộp (Gross Profit)
master['gross_profit'] = master['total_revenue'] - master['total_cogs']

# 4. Biên lợi nhuận (%)
master['profit_margin'] = (master['gross_profit'] / master['total_revenue']) * 100

# Phần 2: Visualization & EDA